# Module 1 — Data Loading and Understanding

**Modular Daraz Analysis — Module 1 of 6**

**Purpose.** Introduce the project, load the raw CSV, inspect structure/data types, and identify the cleaning problems before changing the data.

**Course concepts visible in this part:** `import`, file input with `pd.read_csv()`, variables/DataFrames, data types, strings, missing values, `shape`, `columns`, `info()`, `describe()`, `value_counts()`.

**Project rule:** This notebook is one stage of the same Daraz analysis. It does not use separate practice examples; every concept is applied to the real project data.

## Project introduction

**What we are doing.** We are analysing products sold on **Daraz Pakistan** using a 2024 dataset of top-selling watches, bags and accessories.

**Why we chose this dataset.** Online shopping is familiar, but the dataset is not already analysis-ready. Prices are stored as text, ratings look like `4.6/5`, sold counts look like `6K Sold`, and the voucher column mixes different offer formats. That gives us a realistic reason to use the Python concepts from class for cleaning, grouping, comparison and visualization.

**What we want to find out.**

* which categories have the most product records
* which categories are typically expensive or cheap
* which categories have higher reported sold counts
* whether price, star rating, rating count or discount percentage are related to sold count
* whether products with vouchers or free-delivery offers show different sales patterns
* what the visualizations reveal that is difficult to see from tables alone

Because this is a snapshot dataset, our results describe **associations in this dataset**. They do not prove that price, vouchers, delivery or any other feature causes sales.

## Dataset source and description

**Where it came from.**

| | |
| :---- | :---- |
| Name | Daraz Top Sales Products Dataset 2024 |
| Source | Kaggle, uploaded by muhammadmurtazasani |
| Link | https://www.kaggle.com/datasets/muhammadmurtazasani/daraz-top-sales-products-dataset-2024 |
| File we used | `Watches Bags Accessories.csv` |
| Size | 1078 rows and 11 columns |

**What the columns mean.**

* **Title** is the product name shown on the listing
* **Rating in Stars** is the star rating, but written as text like `4.6/5`
* **Rating Count** is how many people rated the product
* **Sold Count** is how many were sold, written as `452 Sold` or `6K Sold`
* **Voucher** is any voucher offer on the product, and it is the messiest column
* **Delivery** says either `Free Delivery` or just `0`
* **Currency** says `Rs.` on every single row
* **Current Price** is what the product sells for now
* **Original Price** is the price before the discount, written as `Rs. 520`
* **Category** is the type of product, and there are 7 of them
* **Link** is the web address of the product

**Which country is this?** Every link in the file points to `www.daraz.pk`, so this is the
Pakistani site and the prices are in Pakistani Rupees. We checked this rather than assuming,
because Daraz runs separate sites for Pakistan, Bangladesh, Nepal and Sri Lanka and the
currency is different in each one.

## Import libraries
`pandas` is used here because this part needs to load and inspect tabular data. NumPy and Matplotlib are used in later modules where they are needed.

In [1]:
import pandas as pd

## File input — load the dataset

> **Course concepts visible here:** file input, variables and a pandas DataFrame.

`pd.read_csv()` is the file-input step of the project. It reads the CSV file and turns it into a pandas DataFrame stored in the variable `df`. We use `head()` immediately afterward to confirm that the file loaded correctly and to inspect the raw format before changing anything.

In [2]:
df = pd.read_csv("Watches Bags Accessories.csv")
df.head()

,Title,Rating in Stars,Rating Count,Sold Count,Voucher,Delivery,Currency,Current Price,Original Price,Link,Category
0,Yfashion Cartoon Electronic Watch Life Water...,4.6/5,707,6K Sold,0,Free Delivery,Rs.,287,Rs. 520,//www.daraz.pk/products/y-led-3-i258744118-s14...,Kids Watches
1,"100% Imported LED watch for men , boys and Kid...",4.3/5,293,1K Sold,0,0,Rs.,270,Rs. 999,//www.daraz.pk/products/100-led-2023-i40167553...,Kids Watches
2,1 Cartoon Characters Analog Wrist Watch For Ki...,4.5/5,57,452 Sold,0,Free Delivery,Rs.,225,Rs. 750,//www.daraz.pk/products/1-i423737473-s20070599...,Kids Watches
3,M3 Touch LED Bracelet Digital Watch Band Good ...,4.6/5,51,332 Sold,0,Free Delivery,Rs.,160,Rs. 299,//www.daraz.pk/products/m3-led-led-i432456407-...,Kids Watches
4,led Watches For Boys and Girls and kids,4.4/5,54,476 Sold,0,Free Delivery,Rs.,198,Rs. 300,//www.daraz.pk/products/led-i398157365-s192654...,Kids Watches


## Data types and understanding the dataset

> **Course concepts visible here:** variables, strings/text, numeric data types, missing values and DataFrame inspection.

Before cleaning, we inspect the size, column names, data types, missing values and examples of the raw values. This is where the **data-type concept** becomes important: many columns that should be numeric are stored as `object` text, so calculations will not work until we convert them correctly.

In [3]:
# how many rows and columns do we have
df.shape

(1078, 11)

In [4]:
# what are the columns called
df.columns

Index(['Title', 'Rating in Stars', 'Rating Count', 'Sold Count', 'Voucher',
       'Delivery', 'Currency', 'Current Price', 'Original Price', 'Link',
       'Category'],
      dtype='object')

`info()` is the most useful command here because it shows the data type of every column and
how many values are filled in, both at the same time.

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1078 entries, 0 to 1077
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Title            1078 non-null   object
 1   Rating in Stars  1078 non-null   object
 2   Rating Count     1078 non-null   int64 
 3   Sold Count       1044 non-null   object
 4   Voucher          1078 non-null   object
 5   Delivery         1078 non-null   object
 6   Currency         1078 non-null   object
 7   Current Price    1078 non-null   object
 8   Original Price   1010 non-null   object
 9   Link             1078 non-null   object
 10  Category         1078 non-null   object
dtypes: int64(1), object(10)
memory usage: 92.8+ KB


**What we found.** Only `Rating Count` is stored as a number. Every other column is `object`,
which in pandas means text. This is the main problem with the dataset and it is what most of
our cleaning is about.

In [6]:
df.describe()

,Rating Count
count,1078.000000
mean,202.360853
std,429.747239
min,0.000000
25%,25.000000
50%,68.000000
75%,184.750000
max,5553.000000


`describe()` normally gives the average, minimum and maximum of every numeric column. Here it
only shows one column, which proves the point above. Pandas does not see the prices or the
ratings as numbers yet, so it refuses to do any maths on them.

In [7]:
# how many products are in each category
df["Category"].value_counts()

Category
Womens Bags           200
Mens Watches          160
Mens Accessories      160
Womens Watches        160
Kids Watches          158
Mens Bags             120
Womens Accessories    120
Name: count, dtype: int64

In [8]:
# look at the messy looking columns before deciding what to do with them
print(df["Rating in Stars"].unique()[:8])
print(df["Sold Count"].unique()[:8])
print(df["Original Price"].unique()[:5])
print(df["Delivery"].unique())
print(df["Currency"].unique())

['4.6/5' '4.3/5' '4.5/5' '4.4/5' '3.8/5' '4/5' '3.6/5' '3.9/5']
['6K Sold' '1K Sold' '452 Sold' '332 Sold' '476 Sold' '689 Sold' '2K Sold'
 '404 Sold']
['Rs. 520' 'Rs. 999' 'Rs. 750' 'Rs. 299' 'Rs. 300']
['Free Delivery' '0']
['Rs.']


**What this tells us.**

* `Currency` says `Rs.` on all 1078 rows, so it carries no information at all
* `Delivery` uses `0` instead of writing something like "no free delivery", which is
  confusing to read
* the ratings, prices and sold counts all have text mixed into them

In [9]:
# the voucher column is the messiest one, so we look at all of its values
df["Voucher"].value_counts()

Voucher
0              922
3 Vouchers      52
4 Vouchers      33
6 Vouchers      19
2 Vouchers       9
5 Vouchers       8
7 Vouchers       7
8 Vouchers       5
Rs.1500 Off      5
Rs.5 Off         4
9 Vouchers       3
Rs.500 Off       3
Rs.70 Off        2
Rs.20 Off        2
5% Off           1
3% Off           1
Rs.120 Off       1
Rs.100 Off       1
Name: count, dtype: int64

**The problem with `Voucher`.** Three completely different things are stored in this one
column. Some rows count vouchers (`3 Vouchers`), some give an amount off (`Rs.1500 Off`),
some give a percentage (`5% Off`), and most are just `0`.

These cannot be compared with each other. Three vouchers is not bigger or smaller than
Rs 1500 off, they are different kinds of thing. Ananta explains in Module 6 what we decided
to do about it.

## Module 1 output
This part identifies the main data-quality problems that Module 2 will clean: text-based prices and ratings, mixed sold-count formats, ambiguous delivery/voucher values, missing data and repeated product links.